# ML Assignment 2 - Data Preprocessing and Model Training

This notebook prepares the churn dataset, stores raw and processed train/test splits, saves preprocessing artifacts as joblib files, trains the five required classifiers, and saves trained model joblibs for Streamlit.

## 1. Imports and Paths

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SOURCE_DATA_PATH = PROJECT_ROOT / 'data' / 'Churn_Modelling.csv'

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'model'
REPORTS_DIR = PROJECT_ROOT / 'reports'
TEST_DATA_PATH = PROJECT_ROOT / 'test_data.csv'

for directory in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODEL_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## 2. Load Dataset

In [2]:
df = pd.read_csv(SOURCE_DATA_PATH)

print(f'Dataset shape: {df.shape}')
display(df.head())
display(df.info())
display(df['Exited'].value_counts(normalize=True).rename('class_ratio'))

Dataset shape: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


None

Exited
0    0.7963
1    0.2037
Name: class_ratio, dtype: float64

## 3. Define Features and Target

`RowNumber`, `CustomerId`, and `Surname` are identifiers, so they are excluded from model training. The Streamlit upload file should contain the remaining input columns and the `Exited` target column for evaluation.

In [3]:
TARGET_COLUMN = 'Exited'
DROP_COLUMNS = ['RowNumber', 'CustomerId', 'Surname']

feature_columns = [
    column
    for column in df.columns
    if column not in DROP_COLUMNS + [TARGET_COLUMN]
]

categorical_features = ['Geography', 'Gender']
numeric_features = [column for column in feature_columns if column not in categorical_features]

X = df[feature_columns].copy()
y = df[TARGET_COLUMN].copy()

print('Feature columns:', feature_columns)
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Feature columns: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Numeric features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Categorical features: ['Geography', 'Gender']


## 4. Split and Store Raw Data

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_raw = X_train.copy()
train_raw[TARGET_COLUMN] = y_train.values

test_raw = X_test.copy()
test_raw[TARGET_COLUMN] = y_test.values

train_raw.to_csv(RAW_DATA_DIR / 'train_raw.csv', index=False)
test_raw.to_csv(RAW_DATA_DIR / 'test_raw.csv', index=False)
test_raw.to_csv(TEST_DATA_PATH, index=False)

print('Raw train shape:', train_raw.shape)
print('Raw test shape:', test_raw.shape)
print('Saved Streamlit test data to in the workspace as test_data.csv')

Raw train shape: (8000, 11)
Raw test shape: (2000, 11)
Saved Streamlit test data to in the workspace as test_data.csv


## 5. Fit Preprocessor and Store Processed Data

In [7]:
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ],
    remainder='drop',
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
processed_feature_names = preprocessor.get_feature_names_out()

train_processed = pd.DataFrame(X_train_processed, columns=processed_feature_names, index=X_train.index)
train_processed[TARGET_COLUMN] = y_train.values

test_processed = pd.DataFrame(X_test_processed, columns=processed_feature_names, index=X_test.index)
test_processed[TARGET_COLUMN] = y_test.values

train_processed.to_csv(PROCESSED_DATA_DIR / 'train_processed.csv', index=False)
test_processed.to_csv(PROCESSED_DATA_DIR / 'test_processed.csv', index=False)

joblib.dump(preprocessor, MODEL_DIR / 'preprocessor.joblib')
joblib.dump(feature_columns, MODEL_DIR / 'feature_columns.joblib')
joblib.dump(processed_feature_names.tolist(), MODEL_DIR / 'processed_feature_names.joblib')

print('Processed train shape:', train_processed.shape)
print('Processed test shape:', test_processed.shape)
print('Saved preprocessor and feature metadata to model directory')

Processed train shape: (8000, 14)
Processed test shape: (2000, 14)
Saved preprocessor and feature metadata to model directory


## 6. Train Required Models

In [9]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': CalibratedClassifierCV(
        SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_STATE),
        cv=3,
    ),
}

model_files = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Decision Tree': 'decision_tree.joblib',
    'KNN': 'knn.joblib',
    'Naive Bayes': 'naive_bayes.joblib',
    'Support Vector Machine': 'svm.joblib',
}

metrics_rows = []
classification_reports = {}
confusion_matrices = {}

for model_name, model in models.items():
    model.fit(X_train_processed, y_train)

    y_pred = model.predict(X_test_processed)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test_processed)[:, 1]
    else:
        y_score = model.decision_function(X_test_processed)

    metrics_rows.append(
        {
            'ML Model Name': model_name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'AUC': roc_auc_score(y_test, y_score),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1': f1_score(y_test, y_pred, zero_division=0),
            'MCC': matthews_corrcoef(y_test, y_pred),
        }
    )

    classification_reports[model_name] = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0,
    )
    confusion_matrices[model_name] = confusion_matrix(y_test, y_pred).tolist()

    joblib.dump(model, MODEL_DIR / model_files[model_name])
    print(f'Saved {model_name} to model dir {model_files[model_name]}')

Saved Logistic Regression to model dir logistic_regression.joblib
Saved Decision Tree to model dir decision_tree.joblib
Saved KNN to model dir knn.joblib
Saved Naive Bayes to model dir naive_bayes.joblib
Saved Support Vector Machine to model dir svm.joblib


## 7. Store Metrics and Model Registry

In [10]:
metrics_df = pd.DataFrame(metrics_rows).sort_values(by='F1', ascending=False)
metrics_df.to_csv(REPORTS_DIR / 'model_metrics.csv', index=False)

with open(REPORTS_DIR / 'classification_reports.json', 'w') as file:
    json.dump(classification_reports, file, indent=2)

with open(REPORTS_DIR / 'confusion_matrices.json', 'w') as file:
    json.dump(confusion_matrices, file, indent=2)

model_registry = {
    'target_column': TARGET_COLUMN,
    'feature_columns': feature_columns,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'preprocessor_file': 'preprocessor.joblib',
    'feature_columns_file': 'feature_columns.joblib',
    'processed_feature_names_file': 'processed_feature_names.joblib',
    'model_files': model_files,
}

with open(MODEL_DIR / 'model_registry.json', 'w') as file:
    json.dump(model_registry, file, indent=2)

display(metrics_df)
print('Saved metrics to: report directory as model_metrics.csv')
print('Saved model registry to model directory as model_registry.json')

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
4,Support Vector Machine,0.8525,0.854457,0.704380,0.474201,0.566814,0.495693
0,Logistic Regression,0.7135,0.777165,0.387228,0.700246,0.498688,0.348230
2,KNN,0.8350,0.772979,0.663830,0.383292,0.485981,0.417211
1,Decision Tree,0.7770,0.665197,0.454333,0.476658,0.465228,0.324607
3,Naive Bayes,0.8060,0.777307,0.533101,0.375921,0.440922,0.335101


Saved metrics to: report directory as model_metrics.csv
Saved model registry to model directory as model_registry.json
